# Malagasy Roots & Derivations Scraping Project


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd
import re
import urllib3
from IPython.display import display, clear_output

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36"
}

def clean_text(text):
    if not text: return ""
    text = " ".join(text.split()).lower()
    mapping = {'ñ': 'n', 'à': 'a', 'â': 'a', 'è': 'e', 'é': 'e', 'ì': 'i', 'ò': 'o', 'ô': 'o', 'ù': 'u'}
    for char, replacement in mapping.items():
        text = text.replace(char, replacement)
    return text.strip()

## 📚 PARTIE 1 : Scraping des Fototeny (Racines)

In [ ]:
BASE_ROOTS_URL = "https://tenymalagasy.org/bins/rootLists"
ALPHABET = ['a', 'b', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'v', 'z']
CATEGORIES = {
    "Anarana": "posn", 
    "Mpamaritra": "posaj", 
    "Matoantenin'ny atao": "pospv", 
    "Matoantenin'ny mpanao": "posav", 
    "Tambinteny": "posdv",
    "Mpisolo anarana": "pospn",
    "Tenim-piontanana": "posij",
    "Teny hafa": "posu"
}

def scrape_root_page(suffix):
    url = f"{BASE_ROOTS_URL}?o={suffix}"
    try:
        response = requests.get(url, headers=HEADERS, verify=False, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, "html.parser")
        results = {}
        for row in soup.find_all("tr"):
            cols = row.find_all("td")
            if len(cols) < 2: continue
            root_raw = cols[0].get_text(strip=True)
            if "(" in root_raw and root_raw.endswith(")"): continue
            root = clean_text(root_raw)
            words = [clean_text(a.get_text(strip=True)) for a in cols[1].find_all("a")]
            if root and words:
                results[root] = list(set(results.get(root, []) + words))
        return results
    except: return {}

all_roots_dict = {}

def add_to_global_dict(new_data):
    for root, words in new_data.items():
        if root in all_roots_dict:
            all_roots_dict[root] = sorted(list(set(all_roots_dict[root] + words)))
        else:
            all_roots_dict[root] = sorted(words)

print("Scraping par Alphabet...")
for letter in ALPHABET:
    data = scrape_root_page(f"let{letter}")
    add_to_global_dict(data)
    time.sleep(0.1)

print("Scraping par Catégories...")
for name, suffix in CATEGORIES.items():
    data = scrape_root_page(suffix)
    add_to_global_dict(data)
    time.sleep(0.1)

with open('malagasy_roots_final.json', 'w', encoding='utf-8') as f:
    json.dump(all_roots_dict, f, ensure_ascii=False, indent=4)
print(f"✅ {len(all_roots_dict)} racines sauvegardées dans 'malagasy_roots_final.json'")

## 🛠️ PARTIE 2 : Scraping des Ampanteny (Dérivations)

In [ ]:
def get_all_deriv_forms():
    url = "https://tenymalagasy.org/bins/derivLists"
    try:
        r = requests.get(url, headers=HEADERS, verify=False)
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, "html.parser")
        links = soup.find_all("a", href=True)
        forms = []
        for l in links:
            href = l.get("href")
            if "form=" in href:
                f = href.split("form=")[-1]
                if f not in forms: forms.append(f)
        return forms
    except: return []

def scrape_deriv_rule(form):
    url = f"https://tenymalagasy.org/bins/derivLists?form={form}"
    try:
        r = requests.get(url, headers=HEADERS, verify=False, timeout=15)
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, "html.parser")
        links = soup.find_all("a")
        word_links = [l for l in links if l.get('href') and "/bins/teny2/" in l.get('href') and "~" not in l.get_text()]
        
        pairs = []
        for i in range(0, len(word_links) - 1, 2):
            orig = clean_text(word_links[i].get_text(strip=True))
            deriv = clean_text(word_links[i+1].get_text(strip=True))
            if orig and deriv:
                pairs.append({"from": orig, "to": deriv})
        return pairs
    except: return []

In [ ]:
all_forms = get_all_deriv_forms()
print(f"Début du scraping de {len(all_forms)} règles...")

deriv_results = {}
for i, f in enumerate(all_forms):
    pairs = scrape_deriv_rule(f)
    if pairs: deriv_results[f] = pairs
    
    if i % 10 == 0 or i == len(all_forms) - 1:
        clear_output(wait=True)
        print(f"Progression : {i+1}/{len(all_forms)} - {f}")
    time.sleep(0.05)

with open('malagasy_derivations.json', 'w', encoding='utf-8') as file:
    import json
    json.dump(deriv_results, file, ensure_ascii=False, indent=4)
print(f"✅ {len(deriv_results)} règles sauvegardées dans 'malagasy_derivations.json'")

## 📊 Affichage des résultats (Ampanteny)

In [ ]:
import json 
try:
    with open('malagasy_derivations.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Nombre de règles : {len(data)}")
    summary = [{"Règle": r, "Mots": len(p)} for r, p in data.items()]
    display(pd.DataFrame(summary).sort_values("Mots", ascending=False).head(10))
except: pass

## 🔗 PARTIE 3 : Fusion Intelligente (Roots + Derivations)

In [ ]:
import json
import pandas as pd

def fusion_malagasy_data():
    print("Chargement des fichiers...")
    try:
        with open('malagasy_roots_final.json', 'r', encoding='utf-8') as f:
            roots_data = json.load(f)
        with open('malagasy_derivations.json', 'r', encoding='utf-8') as f:
            derivations_data = json.load(f)
    except Exception as e:
        return

    word_to_root = {}
    for root, words in roots_data.items():
        word_to_root[root] = root
        for w in words: word_to_root[w] = root

    fused_dataset, missing_words = [], []
    for rule, pairs in derivations_data.items():
        for pair in pairs:
            orig, dest = pair['from'], pair['to']
            root = word_to_root.get(dest) or word_to_root.get(orig)
            if root:
                fused_dataset.append({"word": dest, "root": root, "rule": rule, "base": orig})
            else:
                missing_words.append({"word": dest, "rule": rule, "base": orig})

    pd.DataFrame(fused_dataset).drop_duplicates().to_csv('malagasy_data_fused.csv', index=False, encoding='utf-8')
    pd.DataFrame(missing_words).drop_duplicates().to_csv('malagasy_extra_words.csv', index=False, encoding='utf-8')

fusion_malagasy_data()

## 🏆 PARTIE 4 : Consolidation Finale (Intégration des Extras)

In [ ]:
import pandas as pd
df_fused = pd.read_csv('malagasy_data_fused.csv')
df_extra = pd.read_csv('malagasy_extra_words.csv')
df_extra['root'] = df_extra['base']
df_final = pd.concat([df_fused, df_extra[['word', 'root', 'rule', 'base']]], ignore_index=True)
df_final = df_final.drop_duplicates().sort_values(by=['word'])
df_final.to_csv('malagasy_data_FINAL.csv', index=False, encoding='utf-8')
print("Fichier 'malagasy_data_FINAL.csv' prêt. ✅")

## 🧬 PARTIE 5 : Décomposition en Morphèmes (ML Ready)

In [10]:
import pandas as pd
import json

def decompose_word(word, rule, base, mapping):
    # Logic récursive pour la décomposition
    # mapping contient d'autres (rule, base) pour d'éventuelles décompositions imbriquées
    
    morphemes = []
    
    if rule == "~~": # Redoublement
        # On décompose la base si possible, sinon on la garde
        base_morphs = get_morphs(base, mapping)
        morphemes = base_morphs + base_morphs
        
    elif "~" in rule:
        parts = rule.split("~")
        
        if len(parts) == 2:
            pref, suff = parts[0], parts[1]
            if pref and not suff: # mi~
                morphemes = [pref] + get_morphs(base, mapping)
            elif not pref and suff: # ~ana
                morphemes = get_morphs(base, mapping) + [suff]
            elif pref and suff: # faha~na
                morphemes = [pref] + get_morphs(base, mapping) + [suff]
        
        elif len(parts) == 3: # Infixe (~om~)
            infix = parts[1]
            # Exemple : base 'hady' -> word 'homady'. Infix 'om' après 'h'
            # On cherche à isoler le reste
            if len(word) > len(base):
                c = base[0]
                rest = base[1:]
                morphemes = [c, infix] + get_morphs(rest, mapping)
            else:
                morphemes = [word]
    else:
        morphemes = [word]
        
    return morphemes

def get_morphs(word, mapping):
    # Si le mot a une règle de décomposition déjà connue, on l'utilise
    if word in mapping:
        rule, base = mapping[word]
        # Pour éviter les boucles infinies ou les erreurs si base == word
        if base != word:
            return decompose_word(word, rule, base, mapping)
    return [word]

def create_morpheme_dataset():
    print("Préparation de la décomposition...")
    df = pd.read_csv('malagasy_data_FINAL.csv')
    
    # Création d'un dictionnaire pour accès rapide aux bases
    mapping = {}
    for _, row in df.iterrows():
        mapping[row['word']] = (row['rule'], row['base'])
    
    final_data = []
    for _, row in df.iterrows():
        word = row['word']
        rule = str(row['rule'])
        base = str(row['base'])
        
        morphemes = decompose_word(word, rule, base, mapping)
        
        final_data.append({
            "word": word,
            "morphemes": json.dumps(morphemes, ensure_ascii=False)
        })
    
    df_morphs = pd.DataFrame(final_data)
    df_morphs.to_csv('malagasy_training_data.csv', index=False, encoding='utf-8')
    
    print("Fichier 'malagasy_training_data.csv' créé avec succès ! ✅")
    display(df_morphs.head(20))

create_morpheme_dataset()

Préparation de la décomposition...
Fichier 'malagasy_training_data.csv' créé avec succès ! ✅


,word,morphemes
0,aada,"[""a"", ""ada""]"
1,aadokadoka,"[""a"", ""adokadoka""]"
2,aainginaingina,"[""a"", ""ainginaingina""]"
3,aakatse,"[""a"", ""akatse""]"
4,aalalana,"[""a"", ""alalana""]"
5,aaloka,"[""a"", ""a"", ""loka""]"
6,aaloke,"[""a"", ""aloke""]"
7,aambe,"[""a"", ""ambe""]"
8,aampatra,"[""a"", ""ampatra""]"
9,aampatse,"[""a"", ""ampatse""]"
